Neste notebook, estarei aplicando autoencoders a tarefa de classificação de imagens.

A rede neural criada neste notebook será treinada para classificar a base de dados MNIST usando imagens 4x8 (36 pixels) em escala cinza (1 canal), ao invés de treinar com as imagens originais do dataset

Essa técnica pode ser usada para reduzir a dimensionalidade de imagens de um dataset caso o dataset esteja demorando muito para treinar nas dimensoes originais.

In [1]:
from torchvision import datasets, transforms
import torch
from torch import nn, optim
from sklearn.metrics import accuracy_score
import time

Criação do dataset:

In [2]:
dataset_train = datasets.MNIST('../', train = True, download = True, 
                               transform = transforms.ToTensor())
loader_train = torch.utils.data.DataLoader(dataset_train, batch_size = 128, 
                                           shuffle = True)

dataset_test = datasets.MNIST('../', train = False, download = True, 
                              transform = transforms.ToTensor())
loader_test = torch.utils.data.DataLoader(dataset_test, batch_size = 128, 
                                          shuffle = True)

Definição da rede neural autoencoder:

In [3]:
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
device

device(type='cuda')

In [4]:
class Autoencoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(nn.Flatten(),
                                   # Codificador
                                   nn.Linear(28*28, 32),
                                   nn.ReLU(),
                                   # Decodificador
                                   nn.Linear(32, 28*28),
                                   nn.Sigmoid())
    def forward(self, X):
        return self.model(X)
    

In [5]:
model = Autoencoder()
model.to(device)

Autoencoder(
  (model): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=784, out_features=32, bias=True)
    (2): ReLU()
    (3): Linear(in_features=32, out_features=784, bias=True)
    (4): Sigmoid()
  )
)

In [6]:
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters())

Faz o treinamento:

In [7]:
for epoch in range(20):
    
    # Treinamento
    total_loss_train = 0.0
    model.train()

    for inputs, _ in loader_train:
        inputs = inputs.to(device)

        optimizer.zero_grad()

        outputs = model(inputs)
        targets = inputs.view(*outputs.shape)
        loss = criterion(outputs, targets)

        loss.backward()
        optimizer.step()

        total_loss_train += loss.item()
    
    # Teste
    total_loss_val = 0.0
    model.eval()

    with torch.no_grad():
        for inputs, _ in loader_test:
            inputs = inputs.to(device)
            outputs = model(inputs)

            targets = inputs.view(*outputs.shape)
            loss = criterion(outputs, targets)
            
            total_loss_val += loss.item()
        
    # Final da época
    print('ÉPOCA {:3d}: perda_train {:.5f} perda_val {:.5f}'.format(epoch + 1, total_loss_train/len(loader_train), total_loss_val/len(loader_test)))

ÉPOCA   1: perda_train 0.24756 perda_val 0.17707
ÉPOCA   2: perda_train 0.15807 perda_val 0.14245
ÉPOCA   3: perda_train 0.13613 perda_val 0.12742
ÉPOCA   4: perda_train 0.12352 perda_val 0.11789
ÉPOCA   5: perda_train 0.11670 perda_val 0.11308
ÉPOCA   6: perda_train 0.11306 perda_val 0.11027
ÉPOCA   7: perda_train 0.11122 perda_val 0.10892
ÉPOCA   8: perda_train 0.11029 perda_val 0.10836
ÉPOCA   9: perda_train 0.10973 perda_val 0.10781
ÉPOCA  10: perda_train 0.10936 perda_val 0.10746
ÉPOCA  11: perda_train 0.10912 perda_val 0.10744
ÉPOCA  12: perda_train 0.10894 perda_val 0.10709
ÉPOCA  13: perda_train 0.10881 perda_val 0.10722
ÉPOCA  14: perda_train 0.10869 perda_val 0.10697
ÉPOCA  15: perda_train 0.10857 perda_val 0.10705
ÉPOCA  16: perda_train 0.10850 perda_val 0.10696
ÉPOCA  17: perda_train 0.10842 perda_val 0.10678
ÉPOCA  18: perda_train 0.10835 perda_val 0.10693
ÉPOCA  19: perda_train 0.10831 perda_val 0.10681
ÉPOCA  20: perda_train 0.10826 perda_val 0.10665


In [8]:
list(model.children())[0]

Sequential(
  (0): Flatten(start_dim=1, end_dim=-1)
  (1): Linear(in_features=784, out_features=32, bias=True)
  (2): ReLU()
  (3): Linear(in_features=32, out_features=784, bias=True)
  (4): Sigmoid()
)

Definição da rede neural encoder:

In [9]:
class Encoder(nn.Module):
    def __init__(self, autoencoder):
        super().__init__()
        # Apenas camadas (0), (1) e (2) do Autoencoder.
        self.model = nn.Sequential(*list(autoencoder.children())[0][:3])

    def forward(self, X):
        return self.model(X)

In [10]:
encoder = Encoder(model)
encoder.to(device)

Encoder(
  (model): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=784, out_features=32, bias=True)
    (2): ReLU()
  )
)

Codificação do dataset:

In [11]:
encoder.eval()

encoded_batches = []
label_batches = []

with torch.no_grad():

    for inputs, labels in loader_train:
        inputs = inputs.to(device)
        outputs = encoder(inputs)

        encoded_batches.append(outputs.detach().cpu())
        label_batches.append(labels.detach().cpu())
    
X_train_encoded = torch.cat(encoded_batches, dim=0)
y_train = torch.cat(label_batches, dim=0)

encoded_train_dataset = torch.utils.data.TensorDataset(X_train_encoded, y_train)
encoded_train_loader = torch.utils.data.DataLoader(encoded_train_dataset, batch_size=128, shuffle=True)


encoded_batches = []
label_batches = []

with torch.no_grad():

    for inputs, labels in loader_test:
        inputs = inputs.to(device)
        outputs = encoder(inputs)

        encoded_batches.append(outputs.detach().cpu())
        label_batches.append(labels.detach().cpu())
    
X_test_encoded = torch.cat(encoded_batches, dim=0)
y_test = torch.cat(label_batches, dim=0)

encoded_test_dataset = torch.utils.data.TensorDataset(X_test_encoded, y_test)
encoded_test_loader = torch.utils.data.DataLoader(encoded_test_dataset, batch_size=128, shuffle=False)

Definição de uma rede neural classificadora que usa o dataset original (não reduzido):

In [12]:
classificadorNormal = nn.Sequential(nn.Linear(784, 397),
                   nn.ReLU(),
                   nn.Linear(397, 10),
                   nn.LogSoftmax(dim = 1)).to(device)

criterion = nn.NLLLoss()
optimizer = optim.Adam(classificadorNormal.parameters())

Treinamento da rede definida acima:

In [13]:
train_start = time.perf_counter()
for epoch in range(20):
    epoch_start = time.perf_counter()
    
    # Treinamento
    total_loss_train = 0.0
    total_accuracy_train = 0.0

    classificadorNormal.train()

    for inputs, labels in loader_train:
        inputs, labels = inputs.to(device), labels.to(device)
        inputs = inputs.view(-1, 28*28)

        optimizer.zero_grad()
        outputs = classificadorNormal(inputs)

        ps = torch.exp(outputs)
        _, top_class = ps.topk(k = 1, dim = 1)

        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss_train += loss.item()
        total_accuracy_train += accuracy_score(labels.detach().cpu().numpy(), 
                                               top_class.detach().cpu().numpy())
    
    # Validação
    total_loss_val = 0.0
    total_accuracy_val = 0.0
    classificadorNormal.eval()

    with torch.no_grad():
        for inputs, labels in loader_test:
            inputs, labels = inputs.to(device), labels.to(device)
            inputs = inputs.view(-1, 28*28)

            outputs = classificadorNormal(inputs)

            ps = torch.exp(outputs)
            _, top_class = ps.topk(k = 1, dim = 1)
            
            loss = criterion(outputs, labels)
            total_loss_val += loss.item()
            total_accuracy_val += accuracy_score(labels.detach().cpu().numpy(), 
                                                 top_class.detach().cpu().numpy())

    epoch_time = time.perf_counter() - epoch_start
    # Final da época
    print('ÉPOCA {:3d}: perda_train {:.5f} precisão_train {:5f} perda_val {:.5f} precisão_val {:5f} tempo {:.2f}s'.format(
            epoch + 1,
            total_loss_train/len(loader_train),
            total_accuracy_train/len(loader_train), 
            total_loss_val/len(loader_test),
            total_accuracy_val/len(loader_test),
            epoch_time))

total_time = time.perf_counter() - train_start
print('Tempo total de treinamento: {:.2f}s'.format(total_time))

ÉPOCA   1: perda_train 0.32913 precisão_train 0.910126 perda_val 0.16576 precisão_val 0.950554 tempo 2.81s
ÉPOCA   2: perda_train 0.13610 precisão_train 0.960776 perda_val 0.10969 precisão_val 0.969047 tempo 3.02s
ÉPOCA   3: perda_train 0.09070 precisão_train 0.973658 perda_val 0.08844 precisão_val 0.971717 tempo 2.80s
ÉPOCA   4: perda_train 0.06570 precisão_train 0.980949 perda_val 0.07600 precisão_val 0.975574 tempo 2.81s
ÉPOCA   5: perda_train 0.04950 precisão_train 0.985680 perda_val 0.07138 precisão_val 0.977057 tempo 2.85s
ÉPOCA   6: perda_train 0.03799 precisão_train 0.989017 perda_val 0.06859 precisão_val 0.977947 tempo 2.97s
ÉPOCA   7: perda_train 0.03032 precisão_train 0.991221 perda_val 0.06087 precisão_val 0.980024 tempo 2.82s
ÉPOCA   8: perda_train 0.02287 precisão_train 0.994142 perda_val 0.06185 precisão_val 0.980320 tempo 2.90s
ÉPOCA   9: perda_train 0.01849 precisão_train 0.994986 perda_val 0.06354 precisão_val 0.981013 tempo 3.06s
ÉPOCA  10: perda_train 0.01369 precis

Definição de uma rede neural classificadora que usa o dataset codificado:

In [14]:
classificadorCodificado = nn.Sequential(nn.Linear(32, 21),
                                        nn.ReLU(),
                                        nn.Linear(21, 10),
                                        nn.LogSoftmax(dim = 1)).to(device)

criterion = nn.NLLLoss()
optimizer = optim.Adam(classificadorCodificado.parameters())

Treinamento da rede definida acima:

In [15]:
train_start = time.perf_counter()
for epoch in range(20):
    epoch_start = time.perf_counter()
    # Treinamento
    total_loss_train = 0.0
    total_accuracy_train = 0.0

    classificadorCodificado.train()

    for inputs, labels in encoded_train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = classificadorCodificado(inputs)

        ps = torch.exp(outputs)
        _, top_class = ps.topk(k = 1, dim = 1)

        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss_train += loss.item()
        total_accuracy_train += accuracy_score(labels.detach().cpu().numpy(), 
                                               top_class.detach().cpu().numpy())
    
    # Validação
    total_loss_val = 0.0
    total_accuracy_val = 0.0
    classificadorCodificado.eval()

    with torch.no_grad():
        for inputs, labels in encoded_test_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            outputs = classificadorCodificado(inputs)

            ps = torch.exp(outputs)
            _, top_class = ps.topk(k = 1, dim = 1)
            
            loss = criterion(outputs, labels)
            total_loss_val += loss.item()
            total_accuracy_val += accuracy_score(labels.detach().cpu().numpy(), 
                                                 top_class.detach().cpu().numpy())

    epoch_time = time.perf_counter() - epoch_start
    # Final da época
    print('ÉPOCA {:3d}: perda_train {:.5f} precisão_train {:5f} perda_val {:.5f} precisão_val {:5f} tempo {:.2f}s'.format(
            epoch + 1,
            total_loss_train/len(encoded_train_loader),
            total_accuracy_train/len(encoded_train_loader), 
            total_loss_val/len(encoded_test_loader),
            total_accuracy_val/len(encoded_test_loader),
            epoch_time))

total_time = time.perf_counter() - train_start
print('Tempo total de treinamento: {:.2f}s'.format(total_time))

ÉPOCA   1: perda_train 1.11504 precisão_train 0.654540 perda_val 0.53937 precisão_val 0.846618 tempo 1.01s
ÉPOCA   2: perda_train 0.47090 precisão_train 0.862629 perda_val 0.40392 precisão_val 0.882318 tempo 1.01s
ÉPOCA   3: perda_train 0.39466 precisão_train 0.883112 perda_val 0.37043 precisão_val 0.890922 tempo 1.04s
ÉPOCA   4: perda_train 0.36383 precisão_train 0.891813 perda_val 0.35065 precisão_val 0.900415 tempo 1.04s
ÉPOCA   5: perda_train 0.34557 precisão_train 0.897088 perda_val 0.34067 precisão_val 0.902591 tempo 1.01s
ÉPOCA   6: perda_train 0.33302 precisão_train 0.900886 perda_val 0.32871 precisão_val 0.904173 tempo 1.04s
ÉPOCA   7: perda_train 0.32093 precisão_train 0.904012 perda_val 0.31047 precisão_val 0.910206 tempo 1.04s
ÉPOCA   8: perda_train 0.31063 precisão_train 0.906850 perda_val 0.30139 precisão_val 0.913370 tempo 0.98s
ÉPOCA   9: perda_train 0.30341 precisão_train 0.910087 perda_val 0.30407 precisão_val 0.914359 tempo 0.97s
ÉPOCA  10: perda_train 0.29741 precis

Pode observar-se que a precisão está ligeiramente menor do que a do classificadorNormal, porém o tempo de treino foi reduzido devido a redução de dimensionalidade do input!